In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [29]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "plate_well" # or None
random_seed = 42

condition_keys = "perturbation" # 数据集perturbation所对应的obs列名
dataset_name = "Sciplex3_llm_test1"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    "cell_line": {
        "type": "categorical",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
    "dose_value": {
        "type": "continuous",
        "transform": "log1p_zscore",
        "control_ot": "global",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "perturbed",
        "contain_in_condition": True,
        "condition_source": "perturbed",
    },
    "time": {
        "type": "continuous",
        "transform": "log1p_zscore",   
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

condition_rep_dict = pd.read_pickle("./data/processed/condition_embeddings_drug_sciplex3_claude_gemini_test1.pkl")
condition_rep_dict = { p : embed["embedding"] for p,embed in condition_rep_dict.items()}

In [19]:
filePath = './data/raw/Sciplex3_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 762795 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [20]:
adata.obs[control_key] = (adata.obs[condition_keys] == "control")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    745217
True      17578
Name: count, dtype: int64


In [21]:
adata.obs[mass_deduct_keys] = adata.obs["plate"].astype(str) + "_" + adata.obs["well"].astype(str) #处理mass

## splitting

In [22]:
rng = np.random.default_rng(random_seed) 
test_ratio = "cellflow"
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 
    print(condition_list)
else:
    # 按condition分割 zeroshot
    if test_ratio == "cellflow":
        test_condition = ["Hesperadin", "TAK-901", "Dacinostat (LAQ824)", "Givinostat (ITF2357)", 
                          "Belinostat (PXD101)", "Quisinostat (JNJ-26481585) 2HCl", "Alvespimycin (17-DMAG) HCl", 
                          "Tanespimycin (17-AAG)", "Flavopiridol HCl"]
    else:
        n_test = max(1, int(len(condition_list) * test_ratio))
        test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(len(train_condition))
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 

['Hesperadin', 'TAK-901', 'Dacinostat (LAQ824)', 'Givinostat (ITF2357)', 'Belinostat (PXD101)', 'Quisinostat (JNJ-26481585) 2HCl', 'Alvespimycin (17-DMAG) HCl', 'Tanespimycin (17-AAG)', 'Flavopiridol HCl']
179
['AG-490 (Tyrphostin B42)', 'Abexinostat (PCI-24781)', 'Alisertib (MLN8237)', 'Busulfan', 'Obatoclax Mesylate (GX15-070)', 'Enzastaurin (LY317615)', 'BMS-265246', 'UNC0379', 'Raltitrexed', 'Tubastatin A HCl', 'BMS-536924', 'AG-14361', 'SRT3025 HCl', 'Iniparib (BSI-201)', 'Lenalidomide (CC-5013)', 'Divalproex Sodium', 'BMS-911543', 'Carmofur', 'Selisistat (EX 527)', 'PJ34', 'Pirarubicin', 'MLN8054', 'Maraviroc', 'Roxadustat (FG-4592)', 'Patupilone (EPO906, Epothilone B)', 'Sodium Phenylbutyrate', 'PD98059', 'Disulfiram', 'Celecoxib', 'Linifanib (ABT-869)', 'Thalidomide', 'SNS-314', 'Glesatinib?(MGCD265)', 'JNJ-26854165 (Serdemetan)', '(+)-JQ1', 'S3I-201', 'AZD1480', 'Droxinostat', 'PF-573228', 'Momelotinib (CYT387)', 'AZ 960', 'Roscovitine (Seliciclib,CYC202)', 'MK-5108 (VX-689)',

In [25]:
del adata

## latent embedding

In [26]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [27]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [28]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[6.4576387  5.0794516  2.015591   1.6251484  1.4992096  1.4569092
 1.3593928  1.2435559  1.1695973  1.0944189  1.0833997  1.1069586
 1.0682032  1.0607481  1.002726   1.0260863  0.96193194 0.99407613
 1.0080111  0.9707628  0.9612596  0.9463771  0.92181647 0.93728083
 0.90902084 0.92681193 0.91206694 0.91688615 0.91206497 0.92697257
 0.8976028  0.89311624 0.89272046 0.89705855 0.8985222  0.8809114
 0.8845827  0.8802581  0.87206876 0.875385   0.8719536  0.85599947
 0.8643734  0.86021507 0.85358405 0.8629807  0.84385014 0.8406527
 0.84647214 0.83976364 0.83095264 0.8397505  0.82418823 0.8254853
 0.82604176 0.8191725  0.8223789  0.82065874 0.8156732  0.8122723
 0.81037194 0.79308146 0.80908537 0.7914552  0.78832173 0.7964639
 0.7750013  0.78972656 0.798569   0.7888105  0.7906255  0.79165965
 0.7730014  0.7666421  0.7857266  0.7845304  0.7763847  0.7633169
 0.777004   0.77584225 0.76455826 0.7541693  0.7588186  0.7556906
 0.750144   0.7491557  0.7559582  0.7413498  0.74336255 0.7293612
 0.73

In [30]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [31]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/Sciplex3_llm_test1_42_cellflow_True_X_pca_100_None
AnnData object with n_obs × n_vars = 17578 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_well', 'cell_line_idx', 'dose_value_scaled', 'time_scaled', 'condition_combined'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'normalized_m', 'pca', 'perturbation_embeddings', 'global_rulebook'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 718049 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type

In [ ]:
adata_control.uns

In [ ]:
adata_train.obs[condition_combined_keys].value_counts()

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()